In [1]:
import pandas as pd
from pathlib import Path

RUTA_PROYECTO = Path("..")
RUTA_PROCESSED = RUTA_PROYECTO / "data" / "procesado"

carpeta_lematizadas = (
    RUTA_PROCESSED / "resenas_lematizadas"
)

carpeta_etiquetadas = (
    RUTA_PROCESSED / "resenas_etiquetadas"
)

carpeta_etiquetadas.mkdir(
    parents=True,
    exist_ok=True
)

archivos_lematizados = sorted(
    carpeta_lematizadas.glob(
        "resenas_lematizadas_*.parquet"
    )
)

print("Archivos encontrados:", len(archivos_lematizados))
print("Carpeta de salida:", carpeta_etiquetadas.resolve())

Archivos encontrados: 47
Carpeta de salida: C:\Users\Paula María\Desktop\IS\5º CURSO\TFG\Trabajo-Fin-de-Grado\data\procesado\resenas_etiquetadas


In [2]:
archivos_anteriores = sorted(
    carpeta_etiquetadas.glob("*.parquet")
)

if archivos_anteriores:
    print(
        "Archivos anteriores encontrados:",
        len(archivos_anteriores)
    )

    for archivo in archivos_anteriores:
        archivo.unlink()

    print("Archivos anteriores eliminados.")

else:
    print("No existen archivos anteriores.")

print(
    "Archivos antes del procesamiento:",
    len(list(carpeta_etiquetadas.glob("*.parquet")))
)


No existen archivos anteriores.
Archivos antes del procesamiento: 0


In [ ]:
# Función para asignar sentimiento
def asignar_sentimiento(puntuacion):
    """
    Genera una etiqueta de sentimiento a partir
    de la puntuación de la reseña.

    1 o 2 estrellas: negativo
    3 estrellas: neutral
    4 o 5 estrellas: positivo
    """

    if puntuacion in [1, 2]:
        return "negativo"

    elif puntuacion == 3:
        return "neutral"

    elif puntuacion in [4, 5]:
        return "positivo"

    else:
        return None

In [ ]:
# Prueba
for puntuacion in [1, 2, 3, 4, 5]:
    print(
        puntuacion,
        ":",
        asignar_sentimiento(puntuacion)
    )

1 : negativo
2 : negativo
3 : neutral
4 : positivo
5 : positivo


In [ ]:
# Aplcación del etiquetado en los datos
total_resenas_etiquetadas = 0
total_etiquetas_nulas = 0
conteo_sentimientos = pd.Series(
    0,
    index=["negativo", "neutral", "positivo"],
    dtype="int64"
)

for numero_bloque, archivo in enumerate(
    archivos_lematizados,
    start=0
):
    bloque = pd.read_parquet(archivo)

    bloque["sentiment"] = (
        bloque["rating"]
        .apply(asignar_sentimiento)
    )

    etiquetas_nulas_bloque = (
        bloque["sentiment"]
        .isna()
        .sum()
    )

    conteo_bloque = (
        bloque["sentiment"]
        .value_counts()
    )

    conteo_sentimientos = conteo_sentimientos.add(
        conteo_bloque,
        fill_value=0
    )

    total_resenas_etiquetadas += len(bloque)
    total_etiquetas_nulas += etiquetas_nulas_bloque

    ruta_salida = carpeta_etiquetadas / (
        f"resenas_etiquetadas_"
        f"{numero_bloque:04d}.parquet"
    )

    bloque.to_parquet(
        ruta_salida,
        index=False
    )

    print(
        f"Bloque {numero_bloque + 1}/"
        f"{len(archivos_lematizados)}: "
        f"{len(bloque):,} reseñas, "
        f"{etiquetas_nulas_bloque:,} etiquetas nulas"
    )

Bloque 1/47: 39,175 reseñas, 0 etiquetas nulas
Bloque 2/47: 32,192 reseñas, 0 etiquetas nulas
Bloque 3/47: 33,664 reseñas, 0 etiquetas nulas
Bloque 4/47: 35,123 reseñas, 0 etiquetas nulas
Bloque 5/47: 32,876 reseñas, 0 etiquetas nulas
Bloque 6/47: 33,926 reseñas, 0 etiquetas nulas
Bloque 7/47: 35,786 reseñas, 0 etiquetas nulas
Bloque 8/47: 40,786 reseñas, 0 etiquetas nulas
Bloque 9/47: 32,918 reseñas, 0 etiquetas nulas
Bloque 10/47: 34,528 reseñas, 0 etiquetas nulas
Bloque 11/47: 37,276 reseñas, 0 etiquetas nulas
Bloque 12/47: 33,484 reseñas, 0 etiquetas nulas
Bloque 13/47: 36,178 reseñas, 0 etiquetas nulas
Bloque 14/47: 33,400 reseñas, 0 etiquetas nulas
Bloque 15/47: 28,976 reseñas, 0 etiquetas nulas
Bloque 16/47: 31,491 reseñas, 0 etiquetas nulas
Bloque 17/47: 31,264 reseñas, 0 etiquetas nulas
Bloque 18/47: 32,306 reseñas, 0 etiquetas nulas
Bloque 19/47: 36,465 reseñas, 0 etiquetas nulas
Bloque 20/47: 26,334 reseñas, 0 etiquetas nulas
Bloque 21/47: 29,921 reseñas, 0 etiquetas nulas
B

In [9]:
#Comprobación resultados

conteo_sentimientos = (
    conteo_sentimientos
    .astype("int64")
)

porcentajes_sentimientos = (
    conteo_sentimientos
    / total_resenas_etiquetadas
    * 100
)

resumen_sentimientos = pd.DataFrame({
    "Sentimiento": conteo_sentimientos.index,
    "Cantidad de reseñas": conteo_sentimientos.to_numpy(),
    "Porcentaje": porcentajes_sentimientos.to_numpy()
})

resumen_sentimientos["Porcentaje"] = (
    resumen_sentimientos["Porcentaje"]
    .round(2).astype(str) + " %"
)

print("\nRESULTADO DE LA GENERACIÓN DE ETIQUETAS")
print("----------------------------------------")
print(
    "Reseñas etiquetadas:",
    f"{total_resenas_etiquetadas:,}"
)
print(
    "Etiquetas nulas:",
    f"{total_etiquetas_nulas:,}"
)

resumen_sentimientos


RESULTADO DE LA GENERACIÓN DE ETIQUETAS
----------------------------------------
Reseñas etiquetadas: 1,485,343
Etiquetas nulas: 0


,Sentimiento,Cantidad de reseñas,Porcentaje
0,negativo,238483,16.06 %
1,neutral,117793,7.93 %
2,positivo,1129067,76.01 %


In [10]:
# Correspondencia

archivo_prueba = pd.read_parquet(
    sorted(
        carpeta_etiquetadas.glob(
            "resenas_etiquetadas_*.parquet"
        )
    )[0]
)

comprobacion_etiquetas = pd.crosstab(
    archivo_prueba["rating"],
    archivo_prueba["sentiment"]
)

comprobacion_etiquetas

sentiment,negativo,neutral,positivo
rating,,,
1,3323,0,0
2,1979,0,0
3,0,3656,0
4,0,0,6093
5,0,0,24124


In [ ]:
# Verificación de etiquetas en todos los archivos
archivos_etiquetados = sorted(
    carpeta_etiquetadas.glob(
        "resenas_etiquetadas_*.parquet"
    )
)

filas_guardadas = 0
etiquetas_nulas_verificadas = 0
etiquetas_incorrectas = 0

for archivo in archivos_etiquetados:
    bloque = pd.read_parquet(
        archivo,
        columns=["rating", "sentiment"]
    )

    filas_guardadas += len(bloque)

    etiquetas_nulas_verificadas += (
        bloque["sentiment"].isna().sum()
    )

    sentimiento_esperado = (
        bloque["rating"]
        .apply(asignar_sentimiento)
    )

    etiquetas_incorrectas += (
        bloque["sentiment"]
        != sentimiento_esperado
    ).sum()

print("\nVERIFICACIÓN FINAL")
print("------------------")
print("Archivos generados:", len(archivos_etiquetados))
print("Filas guardadas:", f"{filas_guardadas:,}")
print(
    "Etiquetas nulas:",
    f"{etiquetas_nulas_verificadas:,}"
)
print(
    "Etiquetas incorrectas:",
    f"{etiquetas_incorrectas:,}"
)


VERIFICACIÓN FINAL
------------------
Archivos generados: 47
Filas guardadas: 1,485,343
Etiquetas nulas: 0
Etiquetas incorrectas: 0
